In [25]:
import requests

url = "https://fie.org/athletes?season=2027&weapon=E&gender=F&category=S&type=I"
headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)

print("Status Code:", response.status_code)
print("Page length:", len(response.text))

Status Code: 200
Page length: 237419


In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, 'html.parser')

# Find the word SANTUCCIO anywhere in the page
name_tag = soup.find(string=lambda t: t and 'SANTUCCIO' in t)

# Print the name of the tag and what it looks like
print("Found inside tag:", name_tag.parent.name)
print("Tag HTML:", name_tag.parent)

Found inside tag: span
Tag HTML: <span class="font-bold uppercase text-base leading-none">SANTUCCIO</span>


In [27]:
# 1. Grab that span
span = soup.find(string=lambda t: t and 'SANTUCCIO' in t).parent

# 2. Look at its parent and grandparent tags
print("Immediate parent:", span.parent.name, span.parent.get('class'))
print("Grandparent:", span.parent.parent.name, span.parent.parent.get('class'))

Immediate parent: span ['notranslate']
Grandparent: div ['max-h-9', 'bg-elevated/50', 'backdrop-blur-lg', 'flex', 'items-center', 'gap-2', 'p-2', 'text-default']


In [30]:
current = soup.find(string=lambda t: t and 'SANTUCCIO' in t).parent

for level in range(1, 8):
    current = current.parent
    if not current:
        break
    snippet = current.get_text(separator=" | ", strip=True)
    # truncate if it gets too long
    display_snippet = (snippet[:100] + '...') if len(snippet) > 100 else snippet
    print(f"Level {level} (<{current.name}> class={current.get('class')}):")
    print(f"   Text: {display_snippet}\n")

Level 1 (<span> class=['notranslate']):
   Text: SANTUCCIO | Alberta

Level 2 (<div> class=['max-h-9', 'bg-elevated/50', 'backdrop-blur-lg', 'flex', 'items-center', 'gap-2', 'p-2', 'text-default']):
   Text: SANTUCCIO | Alberta

Level 3 (<div> class=['absolute', 'bottom-0', 'w-full', 'flex', 'flex-col', 'gap-0.5', 'bg-gradient-to-t', 'from-black/60', 'to-transparent']):
   Text: SANTUCCIO | Alberta | #1 | Women's Epee | Age: | 31 | Hand: | Right

Level 4 (<a> class=[]):
   Text: SANTUCCIO | Alberta | #1 | Women's Epee | Age: | 31 | Hand: | Right

Level 5 (<div> class=['relative', 'group', 'flex', 'shrink-0', 'cursor-pointer', 'flex-col', 'overflow-hidden', 'rounded-md', 'bg-white', 'shadow-md', 'transform-gpu', 'transition-all', 'duration-300', 'ease-in-out', 'hover:shadow-xl', 'lg:w-auto', 'aspect-3/4']):
   Text: Full Women's Epee Rankings | SANTUCCIO | Alberta | #1 | Women's Epee | Age: | 31 | Hand: | Right

Level 6 (<div> class=['grid', 'grid-cols-2', 'md:grid-cols-3', 'gap-4', 'lg

In [31]:
import pandas as pd

# Level 4 was the <a> tag holding the full text
# We can find all links that contain an athlete rank pattern (e.g. '#')
athlete_links = []

for a in soup.find_all('a'):
    text = a.get_text(separator="|", strip=True)
    if "#" in text and "Age:" in text:
        athlete_links.append(a)

print(f"Total athlete cards found: {len(athlete_links)}")

# Let's inspect the text of the first 3 athletes
for idx, a in enumerate(athlete_links[:3], 1):
    print(f"\n--- Athlete #{idx} ---")
    print(a.get_text(separator="\n", strip=True))

Total athlete cards found: 6

--- Athlete #1 ---
SANTUCCIO
Alberta
#1
Women's Epee
Age:
31
Hand:
Right

--- Athlete #2 ---
FAVARETTO
Martina
#1
Women's Foil
Age:
24
Hand:
Right

--- Athlete #3 ---
EGORIAN
Yana
#1
Women's Sabre
Age:
32
Hand:
Left


In [32]:
print("Is MUHARI in text?", "MUHARI" in response.text)
print("Is SONG in text?", "SONG" in response.text)
print("Is LEHIS in text?", "LEHIS" in response.text)

Is MUHARI in text? True
Is SONG in text? True
Is LEHIS in text? True


In [33]:
import re

# Find a 200-character snippet around MUHARI to see the surrounding structure
match = re.search(r'.{0,80}MUHARI.{0,150}', response.text)

if match:
    print("--- Surrounding data snippet ---")
    print(match.group(0))
else:
    print("Not found in regex search.")

--- Surrounding data snippet ---
szter_Hungary.jpg 2x" onerror="this.setAttribute(&#39;data-error&#39;, 1)" alt="MUHARI Eszter" loading="lazy" fetchpriority="auto" class="size-8 rounded-md object-cover object-top shadow-sm max-sm:order-1" src="https://fie.org/cdn-cgi/i


In [34]:
# 1. Find the image for MUHARI
img = soup.find('img', alt=lambda a: a and 'MUHARI' in a)

# 2. Walk up a few parent levels to find the card container
curr = img
for i in range(1, 6):
    curr = curr.parent
    text = curr.get_text(separator=" | ", strip=True)
    if "HUN" in text or "pts" in text or "#2" in text:
        print(f"Level {i} ({curr.name}, class={curr.get('class')}):")
        print("Text:", text)
        print("---")
        break

Level 1 (div, class=['min-w-0', 'flex-1', 'flex', 'flex-wrap', 'items-center', 'gap-x-2', 'gap-y-1']):
Text: MUHARI | Eszter | HUN | 200.000 | pts
---


In [35]:
img = soup.find('img', alt=lambda a: a and 'MUHARI' in a)
card = img.parent.parent  # Level 2

print("Tag:", card.name)
print("Classes:", card.get('class'))
print("Full Card Text:\n", card.get_text(separator=" | ", strip=True))

Tag: span
Classes: ['flex-1', 'flex', 'items-center', 'gap-2', 'text-start', 'break-words']
Full Card Text:
 MUHARI | Eszter | HUN | 200.000 | pts


In [36]:
img = soup.find('img', alt=lambda a: a and 'MUHARI' in a)

# Let's inspect Level 3 and Level 4
card_l3 = img.parent.parent.parent
card_l4 = card_l3.parent

print("Level 3 Text:", card_l3.get_text(separator=" | ", strip=True))
print("Level 4 Text:", card_l4.get_text(separator=" | ", strip=True))

Level 3 Text: # | 2 | MUHARI | Eszter | HUN | 200.000 | pts
Level 4 Text: # | 2 | MUHARI | Eszter | HUN | 200.000 | pts


In [38]:
# 1. Find all athlete thumbnail images in the ranking grid
athlete_imgs = soup.find_all('img', class_=lambda c: c and 'size-8' in c and 'rounded-md' in c)

records = []

for img in athlete_imgs:
    # Level 3 container holds the full row: Rank, Name, Country, Points
    card = img.parent.parent.parent
    tokens = [t.strip() for t in card.get_text(separator="|", strip=True).split("|") if t.strip()]
    
    # We expect a format like: ['#', '2', 'MUHARI', 'Eszter', 'HUN', '200.000', 'pts']
    if len(tokens) >= 6 and tokens[0] == '#':
        rank = tokens[1]
        last_name = tokens[2]
        first_name = tokens[3]
        country = tokens[4]
        points = tokens[5]
        
        records.append({
            "Rank": rank,
            "Name": f"{last_name} {first_name}",
            "Country": country,
            "Points": points
        })

# 2. Convert to DataFrame
df = pd.DataFrame(records)

print(f"Successfully extracted {len(df)} fencers!\n")
print(df.head(10))

df.to_csv("F_epee.csv")

Successfully extracted 16 fencers!

  Rank                       Name Country   Points
0    1          SANTUCCIO Alberta     ITA  200.000
1    2              MUHARI Eszter     HUN  200.000
2    3                  SONG Sera     KOR  197.000
3    4              LEHIS Katrina     EST  190.000
4    5               RIZZI Giulia     ITA  168.000
5    6            HUSISIAN Hadley     USA  148.000
6    7  CANDASSAMY Marie-Florence     FRA  138.000
7    8      LOUIS MARIE Alexandra     FRA  122.000
8    9            MAKSYMENKO Anna     UKR  107.000
9   10                TANG Junyao     CHN  101.750
